In [2]:
from pathlib import Path
import json
import sys
import cv2
import numpy as np

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from implementation import (
    BlinkDetector,
    CalibrationData,
    CalibrationManager,
    ConfidenceFusion,
    EyeAnalyzer,
    MorseDecoder,
    SystemConfig,
    YOLOEyeClassifier,
)


def resolve_path(value: str | Path, base_dir: Path = ROOT) -> Path:
    path = Path(value)
    return path if path.is_absolute() else base_dir / path


def load_runtime_config(config_path: Path):
    raw = json.loads(config_path.read_text(encoding="utf-8"))
    settings = raw.get("settings", {})
    confidence = settings.get("confidence", {})
    timing = settings.get("timing", {})
    ear = settings.get("ear", {})
    model = settings.get("model", {})
    calibration = raw.get("calibration", {})

    config = SystemConfig(
        alpha=float(confidence.get("alpha", 0.4)),
        blink_threshold=float(confidence.get("blink_threshold", 0.5)),
        letter_gap_seconds=float(timing.get("letter_gap_seconds", 1.5)),
        word_gap_seconds=float(timing.get("word_gap_seconds", 3.0)),
        sentence_gap_seconds=float(timing.get("sentence_gap_seconds", 5.0)),
        ear_min=float(ear.get("ear_baseline_closed", ear.get("ear_min", 0.15))),
        ear_max=float(ear.get("ear_baseline_open", ear.get("ear_max", 0.35))),
        smoothing_window=int(confidence.get("smoothing_window", 5)),
        ema_alpha=float(confidence.get("ema_alpha", 0.3)),
        default_blink_duration_ms=float(timing.get("default_blink_duration_ms", 200.0)),
        yolo_model_path=str(resolve_path(model.get("yolo_model_path", SystemConfig().yolo_model_path))),
        use_gpu=bool(model.get("use_gpu", True)),
    )

    calibration_data = CalibrationData(
        is_calibrated=bool(calibration.get("is_calibrated", False)),
        avg_blink_duration_ms=float(calibration.get("avg_blink_duration_ms_threshold", config.default_blink_duration_ms)),
        avg_dot_duration_ms=float(calibration.get("avg_dot_duration_ms", 150.0)),
        avg_dash_duration_ms=float(calibration.get("avg_dash_duration_ms", 400.0)),
        dot_durations=[float(value) for value in calibration.get("dot_durations", [])],
        dash_durations=[float(value) for value in calibration.get("dash_durations", [])],
        ear_baseline_open=float(calibration.get("ear_baseline_open", config.ear_max)),
        ear_baseline_closed=float(calibration.get("ear_baseline_closed", config.ear_min)),
    )

    return raw, config, calibration_data


def run_video_inference(video_path: Path, config_path: Path):
    raw_config, config, calibration_data = load_runtime_config(config_path)

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    video_fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    if not np.isfinite(video_fps) or video_fps <= 0:
        video_fps = float(raw_config.get("capture", {}).get("fps") or config.target_fps)

    eye_analyzer = EyeAnalyzer()
    yolo_classifier = YOLOEyeClassifier(config.yolo_model_path, config.use_gpu)
    confidence_fusion = ConfidenceFusion(config.smoothing_window, config.ema_alpha)
    calibration_manager = CalibrationManager(config)
    calibration_manager.calibration = calibration_data
    blink_detector = BlinkDetector(config, calibration_manager.get_calibration())
    blink_detector.set_fps(video_fps)
    morse_decoder = MorseDecoder(config)

    symbol_tokens = []
    frame_count = 0

    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break

            frame_count += 1
            eye_data, _ = eye_analyzer.process_frame(frame, config)
            if not eye_data.landmarks_detected:
                continue

            yolo_result = yolo_classifier.classify_dual_eye(eye_data.left_crop, eye_data.right_crop)
            fused_confidence = confidence_fusion.fuse(yolo_result, eye_data.normalized_ear, config.alpha)
            smoothed_confidence = confidence_fusion.smooth_ema(fused_confidence)

            blink_event = blink_detector.process(smoothed_confidence)
            if blink_event:
                symbol_tokens.append(blink_event.blink_type.value)
                morse_decoder.add_symbol(blink_event.blink_type.value)

            if blink_detector.is_sentence_gap():
                morse_decoder.process_sentence_gap()
                symbol_tokens.append(" / ")
            elif blink_detector.is_word_gap():
                morse_decoder.process_word_gap()
                symbol_tokens.append(" / ")
            elif blink_detector.is_letter_gap():
                morse_decoder.process_letter_gap()
                symbol_tokens.append(" ")

        morse_decoder.process_letter_gap()

        return {
            "video_path": str(video_path),
            "config_path": str(config_path),
            "video_fps": video_fps,
            "frames_processed": frame_count,
            "morse_symbols": "".join(symbol_tokens).strip(),
            "decoded_text": morse_decoder.get_decoded_text(),
        }
    finally:
        capture.release()
        eye_analyzer.close()


config_path = ROOT / "json" / "goji.json"
video_path = ROOT / "recordings" / "blink_recording_20260425_132908.mp4"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}")

if not video_path.exists():
    raise FileNotFoundError(f"Video file not found: {video_path}")

print(f"Using config: {config_path}")
print(f"Using video: {video_path}")

result = run_video_inference(video_path, config_path)

print(f"Video FPS used: {result['video_fps']:.3f}")
print(f"Frames processed: {result['frames_processed']}")
print(f"Morse sequence: {result['morse_symbols']}")
print("Decoded text:")
print(result["decoded_text"])


ModuleNotFoundError: No module named 'cv2'

In [ ]:
import torch

from run_seq2seq_morse import MorseTokenizer, MorseTransformer, greedy_decode

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = MorseTokenizer()
model = MorseTransformer(len(tokenizer.tokens)).to(DEVICE)
model_path = ROOT / "model_morse_seq2seq.pt"

if not model_path.exists():
    raise FileNotFoundError(f"Seq2Seq model not found: {model_path}")

model.load_state_dict(torch.load(model_path, map_location=DEVICE))

morse_sequence = " ".join(result["morse_symbols"].split())
if not morse_sequence:
    raise ValueError("Morse sequence is empty; run the previous cell first.")

print(f"Normalized morse sequence: {morse_sequence}")

seq2seq_output = greedy_decode(model, morse_sequence, tokenizer, DEVICE)
print("Seq2Seq output:")
print(seq2seq_output)


In [ ]:
_, seq2seq_config, _ = load_runtime_config(config_path)
seq2seq_decoder = MorseDecoder(seq2seq_config)

normalized_seq2seq = " ".join(seq2seq_output.replace("/", " / ").split())
if not normalized_seq2seq:
    raise ValueError("Seq2Seq output is empty; run the previous cell first.")

words = []
current_letters = []
for token in normalized_seq2seq.split():
    if token == "/":
        if current_letters:
            words.append("".join(current_letters))
            current_letters = []
        continue
    char = seq2seq_decoder.decode_sequence(token)
    current_letters.append(char if char else "?")

if current_letters:
    words.append("".join(current_letters))

decoded_seq2seq_text = " ".join(words).strip()
print("Decoded text (from Seq2Seq):")
print(decoded_seq2seq_text)
